# **20 — Cross-Dataset Overlap Analysis**

## Purpose

This notebook evaluates the practical interoperability of the public oncology datasets integrated into this project by quantifying overlap across shared biological and experimental entities.

The analyses performed here focus on:

- shared cell-line coverage across DepMap/CCLE, GDSC, CTRP, and PRISM;
- overlap between molecular profiles, functional dependency datasets, and pharmacogenomic screens;
- lineage representation across overlapping subsets;
- modality-level coverage and sparsity patterns;
- feasibility of downstream lineage-aware integration.

The objective is to establish a structural understanding of dataset compatibility before identifier harmonization and downstream modeling.

## Expected outputs

This notebook generates:

- pairwise overlap summaries;
- shared-entity coverage tables;
- lineage-level overlap distributions;
- modality coverage matrices;
- integration feasibility assessments.

These analyses will guide the harmonization procedures implemented in `phase3_integration`.

---

## Imports and configuration

This section defines the core Python dependencies, project paths, and display settings used throughout the notebook.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 50)
pd.set_option("display.width", 120)

PROJECT_ROOT = Path.cwd().parents[1]  # adjust if needed
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
RESULTS_DIR = PROJECT_ROOT / "reports" / "phase2_eda"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT

## Dataset loading

This section loads dataset-level metadata and annotation files required to evaluate cross-dataset overlap. Large molecular matrices are intentionally avoided unless needed to extract sample or feature identifiers.

In [ ]:
DATASET_FILES = {
    # DepMap / CCLE
    "depmap_model_metadata": DATA_RAW / "depmap" / "Model.csv",
    "depmap_expression": DATA_RAW / "depmap" / "OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv",
    "depmap_crispr": DATA_RAW / "depmap" / "CRISPRGeneEffect.csv",

    # Pharmacogenomics
    "gdsc_response": DATA_RAW / "gdsc" / "gdsc_drug_response.csv",
    "ctrp_response": DATA_RAW / "ctrp" / "ctrp_drug_response.csv",
    "prism_response": DATA_RAW / "prism" / "prism_drug_response.csv",

    # TCGA
    "tcga_metadata": DATA_RAW / "tcga" / "tcga_sample_metadata.csv",

    # LINCS / CMap
    "lincs_signature_metadata": DATA_RAW / "lincs" / "siginfo_beta.txt",
    "lincs_compound_metadata": DATA_RAW / "lincs" / "compoundinfo_beta.txt",
}

In [ ]:
def file_status_table(file_registry: dict[str, Path]) -> pd.DataFrame:
    records = []

    for dataset_name, path in file_registry.items():
        records.append({
            "dataset_object": dataset_name,
            "path": str(path.relative_to(PROJECT_ROOT)),
            "exists": path.exists(),
            "size_mb": round(path.stat().st_size / 1024**2, 2) if path.exists() else np.nan,
        })

    return pd.DataFrame(records)


file_status = file_status_table(DATASET_FILES)
file_status